# 02 · DenPAR Periapical WSP Benchmark (Validation Only)

Evaluates **Track 1 (`denpar_cls`: alveolar bone loss, `cls-lenet2d-v1`)** and **Track 2 (`denpar_seg`: periapical tooth union, `periapical-wsp82-v1`, 82 params)**.

Place the BYOD Zenodo DenPAR dataset outside Git (e.g. Google Drive or `/content/data/denpar`). Held-out test remains sealed.

In [ ]:
from pathlib import Path
import sys

REPO_DIR = Path("/content/dental-wsp")
if REPO_DIR.is_dir() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/WSP_Work/denpar")
from benchmarks.denpar.adapter import verify_denpar_directory
from wsp_core.guards import exposure_guard

exposure_guard(DATA_DIR)
print(verify_denpar_directory(DATA_DIR))

In [ ]:
import pandas as pd
from benchmarks.denpar.adapter import load_denpar_cls, load_denpar_seg
from wsp_core.evaluation.classification import evaluate_classifier
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.schemas import strict_json_loads

# Point to locked responses generated via `dental-wsp lock-reply`
SEG_LOCK = Path("/content/dental-wsp/benchmarks/denpar/locks/denpar_seg.responses.lock.json")
CLS_LOCK = Path("/content/dental-wsp/benchmarks/denpar/locks/denpar_cls.responses.lock.json")

if SEG_LOCK.is_file():
    val_seg = load_denpar_seg(DATA_DIR, split="val")
    lock = strict_json_loads(SEG_LOCK.read_bytes())
    seg_rows = []
    for slot, rec in lock["attempts"].items():
        if rec.get("status") != "valid":
            continue
        sw = evaluate_segmentation_swaps(rec["parameters"], val_seg)
        au = evaluate_segmentation_paired_audit(rec["parameters"], val_seg)
        seg_rows.append({
            "Reply": slot,
            "Intact Dice (%)": sw["intact"]["correct_dice_pct"],
            "Image-only Dice (%)": sw["image_only"]["correct_dice_pct"],
            "Position-only Dice (%)": sw["position_only"]["correct_dice_pct"],
            "Swap loss (pp)": sw["intact"]["swap_loss_pp"],
            "Excluded tooth pixels A (%)": au["excluded_tooth_a_pct"],
        })
    display(pd.DataFrame(seg_rows).set_index("Reply").round(4))

if CLS_LOCK.is_file():
    val_cls = load_denpar_cls(DATA_DIR, split="val")
    lock = strict_json_loads(CLS_LOCK.read_bytes())
    cls_rows = []
    for slot, rec in lock["attempts"].items():
        if rec.get("status") != "valid":
            continue
        res = evaluate_classifier(rec["parameters"], val_cls)
        cls_rows.append({"Reply": slot, **res})
    display(pd.DataFrame(cls_rows).set_index("Reply").round(4))